In [1]:
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd

In [3]:
import sys
sys.path.append('..')
sys.path.append('../..')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# Input

In [4]:
# Articles with predicted products
file = OUTPUT_DIR+'/Articles/filtered_metabolic_eng_articles_with_products_cleaned_norm_V_2025_09_30.json'
clean_prod = pd.read_json(file)

In [5]:
clean_prod.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Doc_text,Product,Norm_product
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,title,production of zeaxanthin and,[zeaxanthin],[zeaxanthin]
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,abstract,heme proteins production,[heme proteins],[hemoprotein]
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,abstract,ethanol production rates,[ethanol],[ethanol]
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,abstract,sterol biosynthesis,[sterol],[sterol]
10653745,A novel genetically engineered pathway for syn...,A new pathway to synthesize poly(hydroxyalkano...,Applied and environmental microbiology,2000,91890.0,10.1128/AEM.66.2.739-743.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:S J,LastName:Liu] [ForeName:A,LastNa...",None,title,A genetically route for production of poly(hyd...,[poly(hydroxyalkanoic-acids)],[pha]


In [7]:
def singularize(word: str) -> str:
    """Convert common plurals to singular form (basic heuristic)."""
    # Avoid false positives like "biomass", "glass"
    if word.endswith(("ss",'sis','gas','virus')):
        return word
    
    if word.endswith("ies") and len(word) > 3:
        return word[:-3] + "y"   # e.g. "batteries" → "battery"
    elif word.endswith("sis") and len(word) > 2:
        return word[:-1]         # e.g. "enzymes" → "enzyme"
    elif word.endswith("s") and len(word) > 1:
        return word[:-1]         # e.g. "lipases" → "lipase"
    return word


In [8]:
import re
antibiotics_list = [
                'megalomicin', 'bacitracin', 'nystatin', 'frenolicin',
                'pelgipeptin', 'valinomycin', 'haliangicin', 'bicyclomycin',
                'myxin', 'pamamycin', 'plipastatin', 'bottromycin',
                'aminoriboflavin', 'roseoflavin', 'nybomycin',
                'pleurotin', 'daptomycin','antimicrobialagent','erythromycina',
                'amphotericin','cephalosporin','subtilin','nargenicina1','microcystin',
                'gentamicinc1a','gentamicinb','gentamicinbgentamicinc1a','avermectinb1a',
                'spinosad','anthracyclineantibiotic','tacrolimusanhydrous','pneumocandin',
                'ansamitocin','ansamitocinp3','polymyxin','tylosin','triostin','colistin',
                'fengycin','aborycin'
            ]
antibiotics_list = antibiotics_list+[n+'s' for n in antibiotics_list]
carotenoids_list = [
                'astaxanthin', 'lycopene', 'zeaxanthin', 'canthaxanthin',
                'violaxanthin', 'cryptoxanthin', 'decaprenoxanthin', 'dopaxanthin',
                'carotene', 'betacarotene', 'alphacarotene', 'gammacarotene',
                'neurosporene', 'phytoene', 'phytofluene', 'lutein','lycopene',
                'capsanthin', 'capsorubin', 'bixin', 'norbixin', 'fucoxanthin',
                'peridinin', 'apocarotenal', 'rhodopin', 'spirilloxanthin','xanthophyll',
                'surfactin','apocarotenoid','c40carotenoid','c50carotenoid','c30carotenoid'
            ]


enzyme_suffixes = [ 'oxidase', 'dehydrogenase', 'transferase', 'hydrolase',
    'lyase', 'isomerase', 'ligase', 'synthetase', 'kinase',
    'phosphatase', 'protease', 'peptidase','lipase','cellulase',
    'lactase','laccase','halyase','phytase','hydrolase','nuclease','carboxylase',
    'amylase','glucanase','aminase'
]


carotenoids_list = carotenoids_list+[n+'s' for n in carotenoids_list]
def change_product(lista):
    new_lista = []
    try:
        for product in lista:
            if product.startswith('bio') and not product.startswith(('biomass','biofilm','biotin')):
                product= product[3:]
            if product.startswith('units'):
                product= product[5:]
                
            # Remove roman numeral prefixes
            for prefix in ['ii', 'iii']:
                if product.startswith(prefix):
                    product = product[len(prefix):]

            # Remove roman numeral suffixes
            for suffix in ['iiii', 'iii', 'ii']:
                if product.endswith(suffix):
                    product = product[:-len(suffix)]

            product = singularize(product)
            # --- Antibiotics ---
    
            if any(product.endswith(suffix) for suffix in enzyme_suffixes):
                new_lista.append('enzyme')
                
            elif product in antibiotics_list:
                new_lista.append('antibiotic')
            elif re.search(r'(mycin|micin|cillin|peptin|plipastatin|cycline|mectin|cidin)$', product):
                new_lista.append('antibiotic')


                
            # --- Carotenoid
            

            elif product in carotenoids_list:
                new_lista.append('carotenoid')
                
            elif product.startswith('vitamin') and len(product)<=10:
                new_lista.append('vitamin')
                
               
            elif product in ['ginsenosidef1','ginsenosiderh1','ginsenosiderh2']:
                new_lista.append('ginsenoside')                  
                
            elif product in ['astaxanthin']:
                new_lista.append('carotenoid')
                
            elif product in ['lactic']:
                new_lista.append('lactate')
            elif product in ['coumaric']:
                new_lista.append('coumarate')
            elif product in ['ascorbic']:
                new_lista.append('ascorbate')
            elif product in ['linolenic']:
                new_lista.append('linolenate')  
            elif product in ['gallic']:
                new_lista.append('gallate')  
                
            elif product in ['tartaric']:
                new_lista.append('tartarate') 
            elif product in ['aminobenzoic']:
                new_lista.append('aminobenzoate')        
                
            elif product in ['carotene']:
                new_lista.append('carotenoid')  
                
            elif product in ['hyaluoronate']:
                new_lista.append('hyaluronate')  
                
            elif product in ['conate']:
                new_lista.append('muconate')  
            elif product in ['ethylenegroup']:
                new_lista.append('ethylene')                  
                
            # --- Main biofuels (keep specific) ---
            elif product in ['ethanol', 'bioethanol','ethanole2g','ethanolabe']:
                new_lista.append('ethanol')
            elif product in ['isobutanol', 'butanol', 'n-butanol']:
                new_lista.append('butanol')
            elif product in ['butanediol','23butanediol', '2,3-butanediol',
                             'meso23butanediol','2r3rbutanediol', 'stereo23butanediol',
                             'butanediolcdw1h1','2r3r23butanediol','butanediol23bdo']:
                new_lista.append('butanediol')
            elif product in ['propanediol', '1,3-propanediol']:
                new_lista.append('propanediol')
                
            # --- Other biofuels (merge into "other biofuel") ---
            elif product in ['fuel', 'biofuels', 'fuels', 'diesel', 'isoprene']:
                new_lista.append('other biofuel')
            elif product.endswith('diol') or product.endswith('anol') or product.endswith('triol'):
                new_lista.append('other biofuel')
                
            # --- Fatty acids ---
            elif product in ['fattyacid', 'fatty acids', 'fatty acid','fattyacids','fatty',
                            'ffas','ffa','fattyacidsffa','microorganismsfattyacidsffa','fa',
                            '3lcpufa','7fa','acidsvfa','cfa','dufa','fattyacidpufa','fattyacidsmcfa',
                             'hfa','lcpufa','mcfa','ocfa','pufa','scfa','ufa','vfa','eicosapentaenoateepa',
                             'epa','pa','vlcpufasdocosadienoate'] or product.endswith('fattyacid'):
                new_lista.append('fatty acid')
                
            # --- Lipids ---
            elif product in ['isoprenoid', 'terpenoid', 'triglyceride']:
                new_lista.append('lipid')
                
            elif product in ['terephthalatetpa', 'tpa']:
                new_lista.append('terephthalate')
              
            # --- Other categories ---
            elif product.startswith(('hydroxybutyratecoa4hbcoa')):
                new_lista.append('4HBCoA')
            elif product.startswith(('4hb', 'poly4hb','poly4hydroxybutyrate')):
                new_lista.append('4HB')
            elif product.startswith(('3-hydroxypropionate', 'hydroxypropionate','poly3hydroxypropionate', '3hp','hp')):
                new_lista.append('3HP')
            elif product in ['polyhydroxyalkanoate','hydroxyalkanoicacid','polyhydroxyalkanote', 'polyhydroxybutyrate','polyr3hydroxybutyrate', 'phb', 'pha','poly3hydroxybutyrate',
                             'polybetahydroxybutyrate','poly3hydroxybutyratep3hb','poly3hydroxyalkanoate','poly3dhydroxybutyrate',
                             'poly3hydroxybutryate','polyhydroxalkanoate','polyhydroxyalkanoatebioplastic','polyhydroxyalkanoatecopolymer',
                             'polyhydroxyalkanoatemclpha','polydroxyalkanoate','polyhydroxyalkanoateterpolymer','polyhydroxyalkanoatephacopolymer',
                             'polyhydroxybetabutyrate','mclpha','sclpha','p3hb','3hb','hydroxybutyrate','entirepoly3hydroxybutyrate',
                             'bioplasticpolyhydroxybutyratep3hb','3hydroxybutyrate','poly‐3‐hydroxybutyrate']:
                new_lista.append('PHB/PHA') 
                
            elif product in ['proteins']:
                new_lista.append('protein')
            elif product in ['cose']:
                new_lista.append('psicose')
            elif product in ['bioalcohol','c5alcohol']:
                new_lista.append('alcohol')

            elif product in ['fucosyllactose2fl']:
                new_lista.append('fucosyllactose')
            elif product in ['nin']:
                new_lista.append('betanin')
            elif product in ['gsh']:
                new_lista.append('glutathione')                
            elif product in ['ep']:
                new_lista.append('exopolysaccharide')
            elif product in ['abe']:
                new_lista.append('acetone')
                new_lista.append('butanol')
                new_lista.append('ethanol')


            elif product in ['dha','docosahexaenoate','docosahexaenoatedha',
                             'dihydroxyacetonedha','dihydroxyacetone']:
                new_lista.append('DHA')
                
            elif product in ['polygammaglutamatepga','polygammaglutamategammapga','polygammaglutamicacid',
                             'pga']:
                new_lista.append('polygammaglutamate')
            # --- Amino acids ---
            elif product in ['aminoacid','aminoacids']:
                new_lista.append('amino acid')
            elif product in [
                'alanine', 'arginine', 'asparagine', 'aspartic acid',
                'cysteine', 'glutamic acid', 'glutamine', 'glycine',
                'histidine', 'isoleucine', 'leucine', 'lysine',
                'methionine', 'phenylalanine', 'proline', 'serine',
                'threonine', 'tryptophan', 'tyrosine', 'valine',
                'glutamate1','aspartate2','ala','phe'
            ]:
                new_lista.append('amino acid')

            elif product in ['transmuconate']:
                new_lista.append('muconate')
            elif product in ['ch4']:
                new_lista.append('methane')
            elif product in ['gaba','aminobutyrategaba','aminobutyrate']:
                new_lista.append('gammaaminobutyrate')                
            elif product in ['gibberellate']:
                new_lista.append('muconate')
            elif product.endswith('ic acid'):
                new_lista.append(product.split('ic acid')[0] + 'ate')
            elif product in ['aminoacid','aminoacids']:
                new_lista.append('amino acid')
            else:
                new_lista.append(product)
    except:
        return None
    return new_lista


In [9]:
categorized = clean_prod
categorized['Categories'] =  clean_prod.Norm_product.apply(change_product)
categorized =categorized.explode('Categories')
#categorized['Categories'] =  standardize_product_names(categorized['Categories'])

grouped_products = categorized.groupby(level=0)['Categories'].agg(list).apply(change_product)



In [10]:
grouped_products = grouped_products.dropna()
grouped_products = grouped_products.apply(lambda x: list(dict.fromkeys(x)))  # preserves order

In [11]:
base_data = categorized.drop(columns=['Categories']).drop_duplicates('Title')
joined_data_rejoined = base_data.join(grouped_products)
joined_data_rejoined.Categories = joined_data_rejoined.Categories.apply(
    lambda x: set(x) if isinstance(x, list) and not (len(x) == 1 and pd.isna(x[0])) else x
)

In [12]:
counts = joined_data_rejoined.Categories.explode().dropna().value_counts()

In [13]:
joined_data_rejoined.columns

Index(['Title', 'Abstract', 'Journal', 'Year', 'PMC_ID', 'DOI', 'Type',
       'Author', 'Text', 'Product_Source', 'Doc_text', 'Product',
       'Norm_product', 'Categories'],
      dtype='object')

In [14]:

joined_data_rejoined

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Doc_text,Product,Norm_product,Categories
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,title,production of zeaxanthin and,[zeaxanthin],[zeaxanthin],{carotenoid}
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,abstract,heme proteins production,[heme proteins],[hemoprotein],{hemoprotein}
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,abstract,ethanol production rates,[ethanol],[ethanol],{ethanol}
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,abstract,sterol biosynthesis,[sterol],[sterol],{sterol}
10653745,A novel genetically engineered pathway for syn...,A new pathway to synthesize poly(hydroxyalkano...,Applied and environmental microbiology,2000,91890.0,10.1128/AEM.66.2.739-743.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:S J,LastName:Liu] [ForeName:A,LastNa...",None,title,A genetically route for production of poly(hyd...,[poly(hydroxyalkanoic-acids)],[pha],{PHB/PHA}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40572067,Synergistic Engineering of the Twin-Arginine T...,Amylosucrase (AS) is a highly versatile enzyme...,Microorganisms,2025,12195555.0,10.3390/microorganisms13061179,"Journal Article,","[ForeName:Caizhe,LastName:Wang] [ForeName:Dand...",Introduction:\nThis study aims to enhance the ...,title,synergistic of the twin-arginine translocation...,[amylosucrase],[amylosucrase],{amylosucrase}
40572208,Heterologous Expression of the Nitrogen-Fixing...,Microbially mediated biological nitrogen fixat...,Microorganisms,2025,12195393.0,10.3390/microorganisms13061320,"Journal Article,","[ForeName:Xiuling,LastName:Wang] [ForeName:Shi...",Introduction:\nThis article reports the first ...,abstract,for the production of nitrogenase,[nitrogenase],[nitrogenase],{nitrogenase}
40573728,Functional Characterization of Squalene Epoxid...,The medicinal plant <i>Siraitia grosvenorii</i...,"Plants (Basel, Switzerland)",2025,NaN,10.3390/plants14121740,"Journal Article,","[ForeName:Huan,LastName:Zhao] [ForeName:Ze,Las...",None,abstract,the plant siraitia grosvenorii the production ...,[cucurbitane-type mogrosides],[cucurbitanetypemogrosides],{cucurbitanetypemogroside}
40577193,Structural characterization and dynamics of Ad...,"<i>Clostridium thermocellum</i>, a cellulolyti...",eLife,2025,NaN,10.7554/eLife.96966,"Journal Article,","[ForeName:Samantha J,LastName:Ziegler] [ForeNa...",None,abstract,the production of ethanol as one of main,[ethanol],[ethanol],{ethanol}


In [15]:
test= joined_data_rejoined.explode('Categories')

In [16]:
counts.index[counts.index.str.endswith('unit')].sort_values()

Index(['45145mm5873unit', '638016unit', 'caseinolyticunit',
       'cphycocyaninholoalphasubunit', 'enzymeunit', 'eunit',
       'ga3oglucose2631unit', 'malonylcoaunit', 'oligomycinstarterunit',
       'polyketidestarterunit', 'putrescine10unitsproline16unit',
       'starterunit', 'subunit', 'trifluoromethylunit'],
      dtype='object', name='Categories')

In [17]:
counts.index[counts.index.str.contains('units')].sort_values()

Index(['30234units1d1', '32u/units·h', '4dihydroxyphenylalanineunitsdopa',
       'cholesterol178unitspregnenolone', 'hbp/unitsdcw/h',
       'lactate61unitswhichproductionbythecontrolstrain',
       'phenylalanineunitspapa', 'putrescine10unitsproline16unit',
       'tartarateunitsta', '·units1'],
      dtype='object', name='Categories')

In [22]:
from normalize_product import *


In [23]:
def compare_words(word1, word2, max_diff=3):
    # Quick reject: if length difference too big
    if abs(len(word1) - len(word2)) > max_diff:
        return False

    # Extra rule: if no letters overlap at all, not similar
    if len(set(word1) & set(word2)) < 2:
        return False

    # Case 1: same length, compare by substitutions
    if len(word1) == len(word2):
        diff_count = 0
        for a, b in zip(word1, word2):
            if a != b:
                diff_count += 1
                if diff_count > max_diff:
                    return False
        return True

    # Case 2: different length, handle insertions
    else:
        if len(word1) < len(word2):
            word1, word2 = word2, word1

        i = j = diff_count = 0
        while i < len(word1) and j < len(word2):
            if word1[i] != word2[j]:
                diff_count += 1
                if diff_count > max_diff:
                    return False
                i += 1
            else:
                i += 1
                j += 1

        while i < len(word1):
            diff_count += 1
            if diff_count > max_diff:
                return False
            i += 1

        return True

In [24]:
words = counts.index.to_list()
for i in range(len(words)):
    for j in range(i+1, len(words)):
        if len(words[i])>1 and len(words[j])>1:

            if compare_words(words[i], words[j]):
                print(f'{words[i]} and {words[j]} are similar')
                #if words[i].endswith('ate') and words[j].endswith('ic'):
                #    print('Acid')
                
                if words[i].endswith(words[j]) or words[j].endswith(words[i]):
                    print('CHEECK')

ethanol and butanol are similar
ethanol and ethanol2a are similar
ethanol and ethanol39 are similar
ethanol and ethanol58 are similar
ethanol and ethanol34 are similar
carotenoid and carotenone are similar
carotenoid and carotene22 are similar
protein and proton are similar
protein and rnaprotein are similar
CHEECK
protein and kteprotein are similar
CHEECK
protein and prenol are similar
protein and proteincfp are similar
protein and proteomic are similar
protein and ldprotein are similar
CHEECK
protein and 093protein are similar
CHEECK
antibiotic and lantibiotic are similar
CHEECK
antibiotic and antibioticbgc are similar
antibiotic and antiemetic are similar
enzyme and coenzyme are similar
CHEECK
enzyme and ltienzyme are similar
CHEECK
enzyme and engine are similar
enzyme and rawenzyme are similar
CHEECK
enzyme and azoenzyme are similar
CHEECK
enzyme and mnpenzyme are similar
CHEECK
enzyme and pppenzyme are similar
CHEECK
enzyme and exoenzyme are similar
CHEECK
enzyme and enzymedna are

h2 and h2o2 are similar
h2 and adh2 are similar
CHEECK
h2 and 2hbp are similar
arbutin and ambrein are similar
arbutin and rutin are similar
arbutin and argyrin are similar
plasmid and plasmiddna are similar
plasmid and amide are similar
sesquiterpenoid and sesquiterpenevoc are similar
hemoglobin and leghemoglobin are similar
CHEECK
anthocyanin and anthocyanidin are similar
aromaticaminoacid and aromaticdaminoacid are similar
ascorbate and ascorbyl are similar
dicarboxylicacid and carboxylicacid are similar
CHEECK
dicarboxylicacid and c4dicarboxylicacid are similar
CHEECK
amorphadiene and amorpha411diene are similar
catechol and catechin are similar
lactonneotetraose and lactontetraose are similar
analog and analogue are similar
analog and algal are similar
analog and santalol are similar
analog and aao are similar
analog and atpanalog are similar
CHEECK
analog and bmaloh are similar
ketoglutarate and ketogluconate are similar
phenol and sterol are similar
phenol and acetol are similar

rna and urea are similar
rna and rocin are similar
rna and sgrna are similar
CHEECK
rna and crrna are similar
CHEECK
rna and rutin are similar
rna and adhr are similar
rna and part are similar
rna and trna are similar
CHEECK
rna and tran are similar
rna and prn are similar
rna and arap are similar
rna and nr are similar
rna and aryl are similar
rna and irone are similar
rna and furan are similar
rna and arop are similar
rna and nrrl are similar
rna and npa are similar
rna and apre are similar
rna and ani are similar
rna and dnr are similar
rna and crna are similar
CHEECK
rna and gern are similar
rna and rtca are similar
rna and rine are similar
rna and mrna are similar
CHEECK
rna and nmr are similar
rna and brunk are similar
rna and srna are similar
CHEECK
rna and 3rac are similar
rna and area are similar
guanosine and adenosine are similar
guanosine and guanine are similar
lain and levan are similar
lain and la are similar
lain and algal are similar
lain and xylan are similar
lain and

coenzyme and coenzymeb12 are similar
coenzyme and coenzymeq8 are similar
sa and sam are similar
sa and sgrna are similar
sa and cas9 are similar
sa and statu are similar
sa and saf are similar
sa and syca are similar
sa and psag9 are similar
sa and suwa are similar
sa and shwan are similar
sa and asbf are similar
sa and ssdna are similar
sa and insea are similar
sa and gas are similar
sa and sda are similar
sa and bpsa are similar
CHEECK
sa and srna are similar
sa and dsdna are similar
phosphate3 and phosphate are similar
propane and orotate are similar
propane and uronate are similar
propane and propor are similar
baicalein1 and baicalin1 are similar
octanoate and retinoate are similar
octanoate and butenoate are similar
octanoate and decanoate are similar
octanoate and nonanoate are similar
tocopherol and tocophreol are similar
fucose and allose are similar
fucose and ribose are similar
fucose and ketose are similar
fucose and fuel/ are similar
fucose and fructose are similar
fucose 

locybin and kocurin are similar
terpineol and terpinene are similar
flavanone and flavan3ol are similar
flavanone and flavonol are similar
polyene and polyyne are similar
polyene and polyp are similar
polyene and polyol are similar
cleosideanalog and cleosideanalogue are similar
macrolactone and macrolactin are similar
macrolactone and pantolactone are similar
heme and hdmf are similar
heme and phage are similar
heme and mhb are similar
heme and meat are similar
heme and hmf are similar
heme and mer are similar
heme and hyper are similar
heme and meso are similar
heme and whose are similar
heme and tech are similar
heme and iethe are similar
heme and cmge are similar
heme and chbe are similar
heme and mec are similar
heme and ether are similar
glycolytic and glycosidic are similar
glycolytic and glycolipid are similar
glycan and lcn1 are similar
glycan and gla are similar
glycan and gcn4 are similar
glycan and gellan are similar
glycopeptide and oligopeptide are similar
glycopeptide an

...

flavone5oglycoside and flavonoidglycoside are similar
lticarbon and oddcarbon are similar
laao and ale are similar
laao and locu are similar
laao and ohad are similar
laao and flax are similar
laao and ota are similar
laao and halo are similar
laao and aao are similar
CHEECK
laao and gluca are similar
laao and hldoa are similar
laao and cal‐a are similar
laao and lac are similar
laao and ldha are similar
laao and welan are similar
laao and coam are similar
laao and calb are similar
teprenone and terpenome are similar
ambroxan and ambroxide are similar
phacar and pacl2 are similar
phacar and tracer are similar
phacar and pca are similar
595mm and 297mm are similar
mannose and lactose are similar
mannose and maltose are similar
50n and 150m3 are similar
50n and c5040 are similar
50n and v350f are similar
dci and tudca are similar
dci and pcd are similar
dci and kind are similar
dci and pdca are similar
dci and dc are similar
dci and cdh are similar
dci and modi are similar
dci and adca a

pmt and tma are similar
pmt and mtr are similar
pmt and ptn are similar
pmt and mt are similar
CHEECK
pmt and tmp are similar
pmt and ctp4 are similar
pmt and pte are similar
pmt and pta are similar
pmt and emcp are similar
pmt and type are similar
pmt and amp are similar
pmt and ptac are similar
pmt and mcp are similar
cucurbitadienol and curcurbitadienol are similar
ale and aryl are similar
ale and gla are similar
ale and peia are similar
ale and pla are similar
ale and apre are similar
ale and flax are similar
ale and pea are similar
ale and allele are similar
ale and faee are similar
ale and vate are similar
ale and halo are similar
ale and final are similar
ale and talose are similar
ale and pacl2 are similar
ale and valid are similar
ale and cal‐a are similar
ale and lac are similar
ale and dgla are similar
ale and ldha are similar
ale and gvalue are similar
ale and tea are similar
ale and calb are similar
ale and area are similar
sulfide and sulfide14 are similar
c26 and c22 are

protongr and protozoan are similar
pdca and dc are similar
pdca and pla are similar
pdca and acp are similar
pdca and cdh are similar
pdca and gada are similar
pdca and adca are similar
pdca and pea are similar
pdca and pta are similar
pdca and dag are similar
pdca and opua are similar
pdca and pbd are similar
pdca and pd are similar
pdca and crna are similar
pdca and cya are similar
pdca and emcp are similar
pdca and cdp are similar
pdca and cp are similar
pdca and rtca are similar
pdca and pdna are similar
pdca and adh2 are similar
pdca and dgla are similar
pdca and ldha are similar
pdca and bpsa are similar
pdca and amp are similar
pdca and kdpg are similar
pdca and pca are similar
pdca and ptac are similar
nrrl and intra are similar
nrrl and crna are similar
nrrl and gern are similar
nrrl and donor are similar
nrrl and mrna are similar
nrrl and nmr are similar
nrrl and srna are similar
ohad and odd are similar
ohad and ota are similar
ohad and dag are similar
ohad and aao are simil

c5040 and c40 are similar
c5040 and c1027 are similar
c5040 and c4c5 are similar
c5040 and c5 are similar
accoa and coq9 are similar
accoa and tobacco are similar
accoa and aao are similar
accoa and cocoa are similar
accoa and diboa are similar
accoa and hldoa are similar
accoa and coam are similar
accoa and comx are similar
accoa and calb are similar
oxalate2 and oxalateoa are similar
pcc7942 and pcc7002 are similar
asbf and sda are similar
asbf and afaa are similar
asbf and fbp are similar
asbf and cscb are similar
ctp4 and gcn4 are similar
ctp4 and pte are similar
ctp4 and c40 are similar
ctp4 and pta are similar
ctp4 and crt are similar
ctp4 and cutin are similar
ctp4 and tc are similar
ctp4 and cdp are similar
ctp4 and cp are similar
ctp4 and rtca are similar
ctp4 and type are similar
ctp4 and c4c5 are similar
ctp4 and pca are similar
ctp4 and ptac are similar
gcn4 and c40 are similar
gcn4 and crna are similar
gcn4 and gern are similar
gcn4 and gluca are similar
gcn4 and cnf are s

valid and ai2 are similar
valid and cal‐a are similar
valid and adh2 are similar
valid and ldha are similar
valid and gvalue are similar
hldoa and lac are similar
hldoa and ldha are similar
hldoa and dsdna are similar
ai2 and adh2 are similar
ai2 and ipa are similar
ai2 and ia1a are similar
rine and mrna are similar
rine and nmr are similar
rine and srna are similar
rine and meric are similar
among and amp are similar
cal‐a and lac are similar
cal‐a and calb are similar
lac and dgla are similar
lac and ldha are similar
lac and welan are similar
lac and pca are similar
lac and calb are similar
ly180 and lpp20 are similar
ly180 and lyze are similar
mrna and nmr are similar
mrna and amp are similar
mrna and srna are similar
mrna and meric are similar
mrna and ma30 are similar
mrna and 3rac are similar
mrna and area are similar
cmge and coam are similar
cmge and comx are similar
cmge and chbe are similar
cmge and mec are similar
cmge and mcp are similar
adh2 and ldha are similar
adh2 and e

In [28]:
import pandas as pd
import numpy as np

# your fake values list (already deduplicated)
fake_values = [
    "towards","difficult","mixture","role","method","standpoint","certain",
    "several","future","overall","degree","suite","procedure","repertoire",
    "recovery","maincatalyst","strategy","discovery","performance",
    "superiorperformance","mechanism","response","interplay","spectrum",
    "candidate","technique","could","proxy","entire","academic","widespread",
    "healthy","costly","despite","dual","myriad","mainobstacle","bilevel",
    "heterotroph","organic","toolbox","wildtype","choice","data","toward",
    "length","survival","criterion","biology","immensevariety","latter",
    "shortcut","frontier","scaleup","bulkscale","grand","common","universal",
    "global","addedvalue","recalcitrance","storage","palette","mode",
    "productivity","machinery","attenuation","redox","phenotypic","fossil",
    "variety","vast","sensor","authentic","revision","scenario","dropin",
    "since","delivery","barrier","portfolio","disease","workhorse","result",
    "phenotype","biotic","mimicry","efficacy","unknown","covert",
    "gatekeeper","native","area","anthropogenic","formula","basic",
    "enantiomeric","problem","cheap","may","chapter","acidic","superior",
    "wealth","scope","benign","ton","juice","proven","meaningful","concise",
    "performer","rand","kind","division","resource","wastewater", "main",
] + ['lignocellulosic','cellulosic','pure','human','synthesis',
     'cell','gene','genes','food','acid','raw','phage','threat','trait','usage',
     'substrate','substance','virus','vitro']

# ensure it's a set for faster lookup
fake_set = set(fake_values)

# function to clean each row (with printing if fake found)
def clean_list_with_logging(values, doc_text, title):
    if not isinstance(values, set):  # skip NaN or non-list
        return values
    
    cleaned = [v for v in values if v not in fake_set]
    cleaned_2 = []
    for clean in cleaned:
        if not (clean.endswith('unit') and len(clean)<7) and not clean[0:-4].isdigit() and not clean.endswith('sis') and not clean.replace('c','').isdigit():
            cleaned_2.append(clean)
            
        else:
            print(clean)
        

    # check if anything was removed
    if len(cleaned_2) != len(values):
        print("⚠️ Found fake value(s)!")
        print("Original list:", values)
        print("Original doc text:", doc_text)
        print("Original title text:", title)
        print("New list:", cleaned_2)

        print("-" * 60)

    return cleaned_2 if cleaned_2 else np.nan

# apply cleaning with both columns
joined_data_rejoined["Acid_antibiotic_normalized_cleaned"] = joined_data_rejoined.apply(
    lambda row: clean_list_with_logging(row["Categories"], row["Doc_text"], row["Title"]),
    axis=1
)

c20
⚠️ Found fake value(s)!
Original list: {'c20'}
Original doc text: production of c20
Original title text: Production of C20 polyunsaturated fatty acids (PUFAs) by pathway engineering: identification of a PUFA elongase component from Caenorhabditis elegans.
New list: []
------------------------------------------------------------
c1027
⚠️ Found fake value(s)!
Original list: {'c1027'}
Original doc text: C-1027 production
Original title text: Biochemical characterization of the SgcA1 alpha-D-glucopyranosyl-1-phosphate thymidylyltransferase from the enediyne antitumor antibiotic C-1027 biosynthetic pathway and overexpression of sgcA1 in Streptomyces globisporus to improve C-1027 production.
New list: []
------------------------------------------------------------
c40
⚠️ Found fake value(s)!
Original list: {'c40'}
Original doc text: produced the structurally fully desaturated C(40) dialdehyde carotenoid 2,4,2,4-tetradehydrolycopendial
Original title text: Identification of a carotenoid o

18
⚠️ Found fake value(s)!
Original list: {'18'}
Original doc text: including production of 18s rna, rrna and 40s ribosome subunit assembly ( dragon and others, 2002 ; granneman and , 2004)
Original title text: Re-evaluation of the impact of <i>BUD21</i> deletion on xylose utilization by <i>Saccharomyces cerevisiae</i>.
New list: []
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: {'acid'}
Original doc text: [ on the production of organic acids
Original title text: [Advances on the production of organic acids by yeast].
New list: []
------------------------------------------------------------
⚠️ Found fake value(s)!
Original list: {'substrate'}
Original doc text: as growth substrate, and conversion to protocatechuic acid
Original title text: Engineering of <i>Rhodococcus jostii</i> RHA1 for utilisation of carboxymethylcellulose.
New list: []
------------------------------------------------------------
homeostasis
⚠️ Found fake value(s)

In [29]:
joined_data_rejoined.Acid_antibiotic_normalized_cleaned.dropna()

10618204                  [carotenoid]
10618209                 [hemoprotein]
10649237                     [ethanol]
10649449                      [sterol]
10653745                     [PHB/PHA]
                       ...            
40572067                [amylosucrase]
40572208                 [nitrogenase]
40573728    [cucurbitanetypemogroside]
40577193                     [ethanol]
40579636          [polygammaglutamate]
Name: Acid_antibiotic_normalized_cleaned, Length: 14229, dtype: object

In [30]:
joined_data_rejoined.Acid_antibiotic_normalized_cleaned.dropna().explode().value_counts().head(60)

Acid_antibiotic_normalized_cleaned
other biofuel         688
ethanol               644
amino acid            561
carotenoid            402
protein               398
antibiotic            378
fatty acid            333
enzyme                300
PHB/PHA               294
lipid                 287
butanol               256
lactate               220
succinate2            213
butanediol            145
3HP                   118
biomass               106
polyketide             92
propanediol            79
acetate                75
xylitol                72
itaconate2             71
malate2                62
riboflavin             61
pyruvate               57
gammaaminobutyrate     57
glycerol               56
fucosyllactose         50
alcohol                49
ester                  49
muconate               48
carbonatom             47
acetoin                46
vitamin                42
resveratrol            41
aminolevulinate        39
vanillin               38
flavonoid              37
but

In [34]:
joined_data_rejoined.dropna(subset=['Acid_antibiotic_normalized_cleaned']).Product_Source.value_counts()

Product_Source
title        8230
abstract     4717
full_text    1282
Name: count, dtype: int64

In [36]:
joined_data_rejoined.to_json('prod.json')

In [1]:
import pandas as pd

In [3]:
products = pd.read_json('prod.json')

In [6]:
data = products.loc[:,['Title','Acid_antibiotic_normalized_cleaned']]

In [7]:
data.columns =['Title','Product_category']

In [8]:
data.to_csv('Product_categorized.csv')